In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

In [ ]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

sim = BasicSimulator()

# Quantum random choice helpers
# Need to choose one of three measurement settings
# Each with probability of a third

u_choice = Operator([
    [1 / math.sqrt(3), -math.sqrt(2 / 3)],
    [math.sqrt(2 / 3), 1 / math.sqrt(3)]
])

prep_1_vs_2 = QuantumCircuit(1, 1)
prep_1_vs_2.unitary(u_choice, [0], label="U_1_2")
prep_1_vs_2.measure(0, 0)

hadamard_coin = QuantumCircuit(1, 1)
hadamard_coin.h(0)
hadamard_coin.measure(0, 0)

# Implements protocol requirement that each part chooses randonly from 3  operators

def quantum_choice_3():
    job = sim.run(transpile(prep_1_vs_2, sim), shots=1, memory=True)
    first = job.result().get_memory()[0]

    if first == '0':
        return 1
    else:
        job2 = sim.run(transpile(hadamard_coin, sim), shots=1, memory=True)
        second = job2.result().get_memory()[0]
        return 2 if second == '0' else 3

# Gives Eve a fair random choice between two bases

def quantum_choice_2():
    job = sim.run(transpile(hadamard_coin, sim), shots=1, memory=True)
    bit = job.result().get_memory()[0]
    return 0 if bit == '0' else 1

# Creates the 2-qubit entangled state:
# Singlet state (|01> - |10>) / sqrt(2)

def singlet_state():
    qc = QuantumCircuit(2)
    qc.x(1)
    qc.h(0)
    qc.cx(0, 1)
    qc.z(0)
    return Statevector.from_instruction(qc)

# Basis rotation + measurement

def rotate_to_measurement_basis(state, qubit, basis):
    qc = QuantumCircuit(2)

    if basis == "Z":
        pass
    elif basis == "X":
        qc.h(qubit)
    elif basis == "W":
        qc.ry(-math.pi / 4, qubit)
    elif basis == "V":
        qc.ry(+math.pi / 4, qubit)
    else:
        raise ValueError("Unknown basis")

    return state.evolve(qc)

# Does one measuremnt in the requested basis

def measure_in_basis(state, qubit, basis):
    rotated = rotate_to_measurement_basis(state, qubit, basis)
    outcome, post = rotated.measure([qubit])

    if isinstance(outcome, str):
        bit = int(outcome)
    else:
        bit = int(outcome[0])

    return bit, post

def bit_to_pm1(bit):
    return +1 if bit == 0 else -1

# Intercept-resend attacker
# Eve measures Bob's qubit in random basis X or Z

def eve_intercept_resend(state):
    eve_basis = "Z" if quantum_choice_2() == 0 else "X"

    if eve_basis == "Z":
        _, post = state.measure([1])
        return post
    else:
        qc1 = QuantumCircuit(2)
        qc1.h(1)
        rotated = state.evolve(qc1)

        _, post = rotated.measure([1])

        qc2 = QuantumCircuit(2)
        qc2.h(1)
        resend_state = post.evolve(qc2)

        return resend_state

# E91 protocol with attacker

def run_e91_attacker(N=200):
    alice_basis = {1: "X", 2: "W", 3: "Z"}
    bob_basis   = {1: "W", 2: "Z", 3: "V"}

    rounds = math.ceil(9 * N / 2)

    alice_key = []
    bob_key = []

    corr_sum = {"XW": 0, "XV": 0, "ZW": 0, "ZV": 0}
    corr_count = {"XW": 0, "XV": 0, "ZW": 0, "ZV": 0}

    for _ in range(rounds):
        i = quantum_choice_3()
        j = quantum_choice_3()

        a_basis = alice_basis[i]
        b_basis = bob_basis[j]

        state = singlet_state()

        # Eve attacks before Alice and Bob measure
        state = eve_intercept_resend(state)

        a_bit, state_after_a = measure_in_basis(state, 0, a_basis)
        b_bit, _ = measure_in_basis(state_after_a, 1, b_basis)

        # Key-generating rounds
        if (i, j) == (2, 1) or (i, j) == (3, 2):
            alice_key.append(a_bit)
            bob_key.append(1 - b_bit)

        # Bell-test rounds
        elif (i, j) == (1, 1):
            corr_sum["XW"] += bit_to_pm1(a_bit) * bit_to_pm1(b_bit)
            corr_count["XW"] += 1

        elif (i, j) == (1, 3):
            corr_sum["XV"] += bit_to_pm1(a_bit) * bit_to_pm1(b_bit)
            corr_count["XV"] += 1

        elif (i, j) == (3, 1):
            corr_sum["ZW"] += bit_to_pm1(a_bit) * bit_to_pm1(b_bit)
            corr_count["ZW"] += 1

        elif (i, j) == (3, 3):
            corr_sum["ZV"] += bit_to_pm1(a_bit) * bit_to_pm1(b_bit)
            corr_count["ZV"] += 1

    E = {}
    for term in ["XW", "XV", "ZW", "ZV"]:
        if corr_count[term] == 0:
            E[term] = 0
        else:
            E[term] = corr_sum[term] / corr_count[term]

    S = abs(E["XW"] - E["XV"] + E["ZW"] + E["ZV"])

    alice_key = alice_key[:N]
    bob_key = bob_key[:N]

    print("E91 WITH ATTACKER")
    print("Key length:", len(alice_key)) # Final key length
    print("Keys match:", alice_key == bob_key) # Shows whether keys match

    # Correlation values

    print("<X ⊗ W> =", E["XW"])
    print("<X ⊗ V> =", E["XV"])
    print("<Z ⊗ W> =", E["ZW"])
    print("<Z ⊗ V> =", E["ZV"])

    # Bell parameter

    print("S =", S)
    print("2*sqrt(2) =", 2 * math.sqrt(2))
    
    print("Alice key (first 30 bits):", alice_key[:30])
    print("Bob key   (first 30 bits):", bob_key[:30])

run_e91_attacker(N=200)

KeyboardInterrupt: 